# Image retrieval preprocessing — 5,000 images
Private dataset; public Qwen3-VL-Embedding-8B + Grounding DINO + SAM 2.1.
20-image smoke test precedes full processing. No captioning. No secrets required.
Resume: attach a previous output dataset and set RESUME_DIR in cell 3.


In [ ]:
import json, os, pathlib, subprocess, sys, traceback
BASE = pathlib.Path('/kaggle/working')
OUT = BASE / 'image_retrieval_output'
OUT.mkdir(exist_ok=True)
CODE = pathlib.Path('/tmp/image_retrieval_code')
CODE.mkdir(exist_ok=True)
(CODE / 'common.py').write_text('"""Portable metadata, atomic checkpoints, colors and output validation (no GPU)."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport os\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(temp, path)\n\n\ndef digest_file(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as source:\n        for chunk in iter(lambda: source.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef build_manifest(data_root, expected=5000):\n    root = Path(data_root)\n    index = {}\n    for path in sorted(root.rglob(\'*.jpg\')):\n        index.setdefault(path.name, []).append(path)\n    rows, ids, used = [], set(), set()\n    with (root / \'metadata_sample.csv\').open(encoding=\'utf-8-sig\', newline=\'\') as source:\n        for raw in csv.DictReader(source):\n            image_id = raw[\'imageID\']\n            if image_id in ids:\n                raise ValueError(f\'Duplicate imageID: {image_id}\')\n            ids.add(image_id)\n            prefix = f"{raw[\'day\']}_{raw[\'owner\']}_"\n            if not image_id.startswith(prefix):\n                raise ValueError(f\'ID/metadata mismatch: {image_id}\')\n            name = image_id[len(prefix):] + \'.jpg\'\n            candidates = index.get(name, [])\n            if len(candidates) != 1:\n                raise ValueError(f\'Expected exactly one image for {image_id}, found {len(candidates)}\')\n            path = candidates[0]\n            if path in used:\n                raise ValueError(f\'Image mapped twice: {name}\')\n            used.add(path)\n            match = re.fullmatch(r\'(\\d+)_(\\d+)\\.jpg\', name)\n            if not match or int(match[1]) != int(raw[\'hour\']) or int(match[2]) != int(raw[\'frame\']):\n                raise ValueError(f\'Hour/frame mismatch: {image_id}\')\n            with Image.open(path) as image:\n                image.load()\n                width, height = image.size\n            classes = json.loads(raw[\'class\'])\n            counts = json.loads(raw[\'class_count\'])\n            if not isinstance(classes, list) or not all(isinstance(x, str) for x in classes):\n                raise ValueError(f\'Invalid classes: {image_id}\')\n            if not isinstance(counts, dict) or not all(isinstance(v, int) and v >= 0 for v in counts.values()):\n                raise ValueError(f\'Invalid counts: {image_id}\')\n            relative = path.relative_to(root)\n            rows.append({**raw, \'hour\': int(raw[\'hour\']), \'frame\': int(raw[\'frame\']),\n                         \'source\': raw[\'owner\'], \'batch\': relative.parts[0],\n                         \'path\': \'images/\' + image_id + \'.jpg\', \'original_path\': relative.as_posix(),\n                         \'width\': width, \'height\': height, \'sha256\': digest_file(path)})\n    if len(rows) != expected or len(used) != len(index) or len(used) != sum(map(len, index.values())):\n        raise ValueError(f\'Incomplete dataset: rows={len(rows)}, files={sum(map(len,index.values()))}, expected={expected}\')\n    return sorted(rows, key=lambda r: (r[\'day\'], r[\'source\'], r[\'hour\'], r[\'frame\'], r[\'imageID\']))\n\n\ndef smoke_indices(rows, count=20):\n    """Round-robin hours, picking evenly spread frames within each hour."""\n    groups = {}\n    for i, row in enumerate(rows):\n        groups.setdefault((row[\'day\'], row[\'source\'], row[\'hour\']), []).append(i)\n    result, round_no = [], 0\n    while len(result) < min(count, len(rows)):\n        changed = False\n        for key in sorted(groups):\n            group = groups[key]\n            if round_no < len(group):\n                # First choose middle, then alternate toward the edges.\n                order = [len(group)//2] + [j for j in range(len(group)) if j != len(group)//2]\n                result.append(group[order[round_no]])\n                changed = True\n                if len(result) >= min(count, len(rows)):\n                    break\n        if not changed:\n            break\n        round_no += 1\n    return result\n\n\ndef color_distribution(rgb, mask):\n    """Heuristic HSV palette over foreground only; retains all color fractions."""\n    pixels = np.asarray(rgb, dtype=np.uint8)\n    mask = np.asarray(mask, dtype=bool)\n    if mask.shape != pixels.shape[:2] or not mask.any():\n        raise ValueError(\'Mask must have foreground pixels and match image dimensions\')\n    hsv = np.asarray(Image.fromarray(pixels).convert(\'HSV\'), dtype=np.float32)[mask]\n    h, s, v = hsv[:, 0] * (360 / 255), hsv[:, 1] / 255, hsv[:, 2] / 255\n    names = np.full(len(h), \'gray\', dtype=\'<U8\')\n    names[(s < .18) & (v > .78)] = \'white\'\n    chroma = s >= .18\n    for name, low, high in [(\'red\',0,15),(\'orange\',15,45),(\'yellow\',45,70),(\'green\',70,165),\n                            (\'cyan\',165,195),(\'blue\',195,260),(\'purple\',260,315),(\'pink\',315,345),(\'red\',345,361)]:\n        names[chroma & (h >= low) & (h < high)] = name\n    names[chroma & (h >= 15) & (h < 50) & (v < .65)] = \'brown\'\n    names[v < .18] = \'black\'\n    unique, counts = np.unique(names, return_counts=True)\n    return dict(sorted(((str(n), float(c / len(names))) for n, c in zip(unique, counts)), key=lambda x: -x[1]))\n\n\ndef validate_vectors(vectors, count, dim=4096):\n    if vectors.shape != (count, dim):\n        raise ValueError(f\'Unexpected vector shape: {vectors.shape}\')\n    for start in range(0, count, 256):\n        part = np.asarray(vectors[start:start+256], dtype=np.float32)\n        if not np.isfinite(part).all() or not np.allclose(np.linalg.norm(part, axis=1), 1, atol=1e-4):\n            raise ValueError(f\'Invalid/non-normalized vectors near row {start}\')\n\n\ndef read_embedding_shard(path, ids, dim):\n    with np.load(path, allow_pickle=False) as shard:\n        if shard[\'ids\'].tolist() != ids:\n            raise ValueError(f\'Checkpoint IDs mismatch: {path}\')\n        vectors = shard[\'vectors\'].astype(np.float32)\n    validate_vectors(vectors, len(ids), dim)\n    return vectors\n', encoding='utf-8')
(CODE / 'pipeline.py').write_text('"""Kaggle GPU worker. Public checkpoints only; no captioning or API credentials."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.metadata\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nfrom common import (atomic_json, color_distribution, digest_file, fingerprint,\n                    read_embedding_shard, smoke_indices, validate_vectors)\n\nACTIVE_IMAGE = None\n\n\ndef write_parquet(path, rows, schema=None):\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    path = Path(path)\n    tmp = path.with_suffix(\'.tmp.parquet\')\n    pq.write_table(pa.Table.from_pylist(rows, schema=schema), tmp)\n    os.replace(tmp, path)\n\n\ndef object_schema():\n    import pyarrow as pa\n    return pa.schema([\n        (\'imageID\', pa.string()), (\'instance_id\', pa.string()), (\'label\', pa.string()),\n        (\'confidence\', pa.float32()), (\'bbox_xyxy\', pa.list_(pa.float32())),\n        (\'mask_path\', pa.string()), (\'mask_sha256\', pa.string()),\n        (\'mask_quality\', pa.float32()), (\'mask_area\', pa.int64()),\n        (\'dominant_color\', pa.string()), (\'colors_json\', pa.string()),\n    ])\n\n\ndef load_rows(root):\n    rows = [json.loads(line) for line in (root / \'manifest.jsonl\').read_text(encoding=\'utf-8\').splitlines() if line]\n    for row in rows:\n        relative = Path(row[\'path\'])\n        resolved = (root / relative).resolve()\n        if relative.is_absolute() or not resolved.is_relative_to(root.resolve()):\n            raise ValueError(\'Unsafe image path in manifest\')\n    if len({r[\'imageID\'] for r in rows}) != len(rows):\n        raise ValueError(\'Duplicate image IDs\')\n    return rows\n\n\ndef gpu_report():\n    import torch\n    return [{\'name\': torch.cuda.get_device_name(i),\n             \'total_gib\': round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2),\n             \'peak_allocated_gib\': round(torch.cuda.max_memory_allocated(i) / 2**30, 3),\n             \'peak_reserved_gib\': round(torch.cuda.max_memory_reserved(i) / 2**30, 3)}\n            for i in range(torch.cuda.device_count())]\n\n\nclass QwenEncoder:\n    """Official Qwen embedding wrapper with explicit multi-GPU loading.\n\n    The official wrapper calls .to(cuda) after from_pretrained, which would\n    collapse a sharded model. Instantiate its component class directly instead.\n    Query code must reuse this class, the pinned config and its instruction.\n    """\n    def __init__(self, cfg):\n        import torch\n        from huggingface_hub import hf_hub_download\n        from transformers import AutoProcessor\n        path = hf_hub_download(cfg[\'model_id\'], \'scripts/qwen3_vl_embedding.py\', revision=cfg[\'revision\'], token=False)\n        spec = importlib.util.spec_from_file_location(\'pinned_qwen_embedding\', path)\n        module = importlib.util.module_from_spec(spec)\n        sys.modules[spec.name] = module\n        spec.loader.exec_module(module)\n        self.helper_sha256 = digest_file(path)\n        self.cfg = cfg\n        self.helper = module\n        self.model = module.Qwen3VLForEmbedding.from_pretrained(\n            cfg[\'model_id\'], revision=cfg[\'revision\'], token=False,\n            torch_dtype=torch.float16, device_map=\'balanced\',\n            max_memory={0: \'13GiB\', 1: \'13GiB\'}, attn_implementation=\'sdpa\',\n        ).eval()\n        device_map = self.model.hf_device_map\n        devices = {str(x) for x in device_map.values()}\n        if not {\'0\', \'1\'}.issubset(devices) and not {\'cuda:0\', \'cuda:1\'}.issubset(devices):\n            raise RuntimeError(f\'Expected sharding on two GPUs, got {device_map}\')\n        if devices & {\'cpu\', \'disk\'}:\n            raise RuntimeError(\'Model does not fit on two GPUs without offload\')\n        self.processor = AutoProcessor.from_pretrained(cfg[\'model_id\'], revision=cfg[\'revision\'], token=False, padding_side=\'right\')\n        # Formatting only; do not invoke the official single-GPU constructor.\n        self.formatter = module.Qwen3VLEmbedder.__new__(module.Qwen3VLEmbedder)\n        for key in (\'min_pixels\', \'max_pixels\', \'max_length\'):\n            setattr(self.formatter, key, cfg[key])\n        self.formatter.default_instruction = cfg[\'instruction\']\n        self.formatter.total_pixels = cfg[\'max_pixels\'] * 10\n        self.formatter.fps = 1\n        self.formatter.max_frames = 64\n\n    def encode(self, items):\n        import torch\n        from qwen_vl_utils import process_vision_info\n        conversations = [self.formatter.format_model_input(**item) for item in items]\n        texts = self.processor.apply_chat_template(conversations, add_generation_prompt=True, tokenize=False)\n        # Raise on preprocessing failure; never silently embed a NULL document.\n        images, videos = process_vision_info(conversations, image_patch_size=16)\n        inputs = self.processor(text=texts, images=images, videos=videos, padding=True,\n                                truncation=True, max_length=self.cfg[\'max_length\'],\n                                do_resize=False, return_tensors=\'pt\')\n        inputs = inputs.to(self.model.get_input_embeddings().weight.device)\n        with torch.inference_mode():\n            outputs = self.model(**inputs, use_cache=False)\n            vector = self.helper.Qwen3VLEmbedder._pooling_last(\n                outputs.last_hidden_state, inputs[\'attention_mask\'].to(outputs.last_hidden_state.device))\n            vector = torch.nn.functional.normalize(vector.float(), p=2, dim=-1)\n        result = vector.cpu().numpy()\n        validate_vectors(result, len(items), self.cfg[\'dimension\'])\n        return result\n\n\ndef embedding_worker(root, out, rows, config, smoke):\n    global ACTIVE_IMAGE\n    cfg = config[\'embedding\']\n    shard_dir = out / \'checkpoints\' / \'embeddings\'\n    shard_dir.mkdir(parents=True, exist_ok=True)\n    indices = smoke_indices(rows, config[\'smoke_images\']) if smoke else list(range(len(rows)))\n    pending = []\n    # Per-image smoke shards are also reused by the full pass.\n    for i in indices:\n        path = shard_dir / f\'{i:06d}.npz\'\n        if path.exists():\n            read_embedding_shard(path, [rows[i][\'imageID\']], cfg[\'dimension\'])\n        else:\n            pending.append(i)\n    start = time.monotonic()\n    encoder = None\n    if pending or smoke:\n        encoder = QwenEncoder(cfg)\n        atomic_json(out / \'embedding_runtime.json\', {\n            **cfg, \'helper_sha256\': encoder.helper_sha256,\n            \'device_map\': {k: str(v) for k, v in encoder.model.hf_device_map.items()},\n            \'pooling\': \'last_non_padding_token\', \'normalization\': \'L2\', \'output_dtype\': \'float32\',\n        })\n    for done, i in enumerate(pending, 1):\n        ACTIVE_IMAGE = rows[i][\'imageID\']\n        vector = encoder.encode([{\'image\': str((root / rows[i][\'path\']).resolve())}])\n        path = shard_dir / f\'{i:06d}.npz\'\n        temp = path.with_suffix(\'.tmp.npz\')\n        np.savez(temp, ids=np.array([rows[i][\'imageID\']]), vectors=vector)\n        os.replace(temp, path)\n        if done % cfg[\'checkpoint_size\'] == 0 or done == len(pending):\n            print(f\'Embedding: {done}/{len(pending)} newly processed\', flush=True)\n    ACTIVE_IMAGE = None\n    if smoke:\n        queries = [\'A person washing a cup at the kitchen sink.\', \'People sitting around a dining table.\']\n        query_vectors = encoder.encode([{\'text\': text} for text in queries])\n        sample = np.concatenate([read_embedding_shard(shard_dir / f\'{i:06d}.npz\', [rows[i][\'imageID\']], cfg[\'dimension\']) for i in indices])\n        scores = query_vectors @ sample.T\n        atomic_json(out / \'smoke_queries.json\', [\n            {\'query\': query, \'results\': [{\'imageID\': rows[indices[j]][\'imageID\'], \'score\': float(scores[q, j])}\n                                        for j in np.argsort(-scores[q])[:5]]}\n            for q, query in enumerate(queries)\n        ])\n    else:\n        temp = out / \'image_embeddings.tmp.npy\'\n        vectors = np.lib.format.open_memmap(temp, mode=\'w+\', dtype=\'float32\', shape=(len(rows), cfg[\'dimension\']))\n        for i, row in enumerate(rows):\n            vectors[i] = read_embedding_shard(shard_dir / f\'{i:06d}.npz\', [row[\'imageID\']], cfg[\'dimension\'])[0]\n        vectors.flush()\n        validate_vectors(vectors, len(rows), cfg[\'dimension\'])\n        del vectors\n        os.replace(temp, out / \'image_embeddings.npy\')\n        atomic_json(out / \'embedding_ids.json\', [r[\'imageID\'] for r in rows])\n    atomic_json(out / (\'smoke_embedding.json\' if smoke else \'embedding_metrics.json\'),\n                {\'images\': len(indices), \'new_images\': len(pending), \'seconds\': time.monotonic()-start, \'gpus\': gpu_report()})\n\n\nclass ObjectEncoder:\n    def __init__(self, config):\n        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Processor, Sam2Model\n        self.cfg = config[\'detection\']\n        dc, sc = config[\'detection\'], config[\'segmentation\']\n        self.det_processor = AutoProcessor.from_pretrained(dc[\'model_id\'], revision=dc[\'revision\'], token=False)\n        self.detector = AutoModelForZeroShotObjectDetection.from_pretrained(dc[\'model_id\'], revision=dc[\'revision\'], token=False).to(\'cuda:0\').eval()\n        self.seg_processor = Sam2Processor.from_pretrained(sc[\'model_id\'], revision=sc[\'revision\'], token=False)\n        self.segmenter = Sam2Model.from_pretrained(sc[\'model_id\'], revision=sc[\'revision\'], token=False).to(\'cuda:1\').eval()\n\n    def detect(self, image, labels):\n        import torch\n        from torchvision.ops import batched_nms\n        detections = []\n        n = self.cfg[\'labels_per_prompt\']\n        for start in range(0, len(labels), n):\n            chunk = labels[start:start+n]\n            inputs = self.det_processor(images=image, text=\'. \'.join(chunk) + \'.\', return_tensors=\'pt\').to(\'cuda:0\')\n            with torch.inference_mode():\n                outputs = self.detector(**inputs)\n            results = self.det_processor.post_process_grounded_object_detection(\n                outputs, inputs.input_ids, threshold=self.cfg[\'box_threshold\'],\n                text_threshold=self.cfg[\'text_threshold\'], target_sizes=[image.size[::-1]])[0]\n            for box, score, label in zip(results[\'boxes\'].cpu().tolist(), results[\'scores\'].cpu().tolist(), results[\'text_labels\']):\n                label = label.strip().lower().rstrip(\'.\')\n                # Phrase decoding can concatenate labels: do not invent a class.\n                if label in chunk and box[2] > box[0] and box[3] > box[1]:\n                    detections.append({\'bbox_xyxy\': box, \'confidence\': score, \'label\': label})\n        if not detections:\n            return []\n        boxes = torch.tensor([x[\'bbox_xyxy\'] for x in detections])\n        scores = torch.tensor([x[\'confidence\'] for x in detections])\n        label_ids = torch.tensor([labels.index(x[\'label\']) for x in detections])\n        keep = batched_nms(boxes, scores, label_ids, self.cfg[\'nms_iou\']).tolist()\n        return [detections[i] for i in keep]\n\n    def masks(self, image, detections):\n        import torch\n        n = self.cfg[\'mask_batch_size\']\n        for start in range(0, len(detections), n):\n            chunk = detections[start:start+n]\n            inputs = self.seg_processor(images=image, input_boxes=[[x[\'bbox_xyxy\'] for x in chunk]], return_tensors=\'pt\').to(\'cuda:1\')\n            with torch.inference_mode():\n                output = self.segmenter(**inputs, multimask_output=False)\n            masks = self.seg_processor.post_process_masks(output.pred_masks.cpu(), inputs[\'original_sizes\'].cpu())[0]\n            qualities = output.iou_scores.detach().cpu().reshape(-1).tolist()\n            for detection, mask, quality in zip(chunk, masks, qualities):\n                yield detection, mask.squeeze(0).numpy().astype(bool), quality\n\n\ndef read_object_checkpoint(path, row, out):\n    result = json.loads(path.read_text(encoding=\'utf-8\'))\n    if result[\'imageID\'] != row[\'imageID\'] or result[\'image_sha256\'] != row[\'sha256\']:\n        raise ValueError(f\'Object checkpoint mismatch: {path}\')\n    for obj in result[\'objects\']:\n        mask_path = (out / obj[\'mask_path\']).resolve()\n        if not mask_path.is_relative_to(out.resolve()) or not mask_path.is_file() or digest_file(mask_path) != obj[\'mask_sha256\']:\n            raise ValueError(f\'Missing/corrupt mask in checkpoint: {path}\')\n    return result[\'objects\']\n\n\ndef object_worker(root, out, rows, config, smoke):\n    global ACTIVE_IMAGE\n    indices = smoke_indices(rows, config[\'smoke_images\']) if smoke else list(range(len(rows)))\n    checkpoints = out / \'checkpoints\' / \'objects\'\n    checkpoints.mkdir(parents=True, exist_ok=True)\n    masks_dir = out / \'masks\'\n    masks_dir.mkdir(exist_ok=True)\n    pending = []\n    for i in indices:\n        path = checkpoints / f\'{i:06d}.json\'\n        if path.exists():\n            read_object_checkpoint(path, rows[i], out)\n        else:\n            pending.append(i)\n    start, object_count = time.monotonic(), 0\n    encoder = ObjectEncoder(config) if pending else None\n    for done, i in enumerate(pending, 1):\n        row = rows[i]\n        ACTIVE_IMAGE = row[\'imageID\']\n        with Image.open(root / row[\'path\']) as opened:\n            image = opened.convert(\'RGB\')\n        detections = encoder.detect(image, config[\'labels\'])\n        objects = []\n        for j, (det, mask, quality) in enumerate(encoder.masks(image, detections)):\n            if not mask.any():\n                continue\n            instance = f"{row[\'imageID\']}__{j:03d}"\n            mask_path = masks_dir / f\'{instance}.png\'\n            Image.fromarray(mask.astype(\'uint8\') * 255).save(mask_path)\n            colors = color_distribution(image, mask)\n            objects.append({**det, \'imageID\': row[\'imageID\'], \'instance_id\': instance,\n                            \'mask_path\': mask_path.relative_to(out).as_posix(), \'mask_sha256\': digest_file(mask_path),\n                            \'mask_quality\': float(quality), \'mask_area\': int(mask.sum()),\n                            \'dominant_color\': next(iter(colors)), \'colors_json\': json.dumps(colors)})\n        atomic_json(checkpoints / f\'{i:06d}.json\', {\'imageID\': row[\'imageID\'], \'image_sha256\': row[\'sha256\'], \'objects\': objects})\n        if done % 10 == 0 or done == len(pending):\n            print(f\'Object/color: {done}/{len(pending)} newly processed\', flush=True)\n    ACTIVE_IMAGE = None\n    all_objects = []\n    for i in indices:\n        objects = read_object_checkpoint(checkpoints / f\'{i:06d}.json\', rows[i], out)\n        object_count += len(objects)\n        if not smoke:\n            all_objects.extend(objects)\n        else:\n            make_overlay(root, out, rows[i], objects)\n    if smoke and object_count == 0:\n        raise RuntimeError(\'Smoke test detected zero objects; inspect vocabulary and thresholds\')\n    if not smoke:\n        write_parquet(out / \'objects.parquet\', all_objects, object_schema())\n    atomic_json(out / (\'smoke_objects.json\' if smoke else \'object_metrics.json\'),\n                {\'images\': len(indices), \'new_images\': len(pending), \'objects\': object_count,\n                 \'seconds\': time.monotonic()-start, \'gpus\': gpu_report()})\n\n\ndef make_overlay(root, out, row, objects):\n    with Image.open(root / row[\'path\']) as image:\n        image = image.convert(\'RGB\')\n    pixels = np.asarray(image).copy()\n    for obj in objects:\n        with Image.open(out / obj[\'mask_path\']) as mask_image:\n            mask = np.asarray(mask_image) > 0\n        pixels[mask] = (pixels[mask].astype(float) * .7 + np.array([0, 255, 100]) * .3).astype(\'uint8\')\n    image = Image.fromarray(pixels)\n    draw = ImageDraw.Draw(image)\n    for obj in objects:\n        draw.rectangle(obj[\'bbox_xyxy\'], outline=\'red\', width=2)\n        draw.text(tuple(obj[\'bbox_xyxy\'][:2]), f"{obj[\'label\']} {obj[\'dominant_color\']} {obj[\'confidence\']:.2f}", fill=\'red\', stroke_width=1, stroke_fill=\'white\')\n    image.thumbnail((1000, 750))\n    directory = out / \'smoke_overlays\'\n    directory.mkdir(exist_ok=True)\n    image.save(directory / (row[\'imageID\'] + \'.jpg\'))\n\n\ndef preflight(config):\n    import torch\n    from huggingface_hub import get_hf_file_metadata, hf_hub_url\n    if torch.cuda.device_count() != 2 or any(\'T4\' not in torch.cuda.get_device_name(i) for i in range(2)):\n        raise RuntimeError(\'This run requires exactly 2 NVIDIA T4 GPUs; no automatic hardware/model fallback\')\n    for group in (\'embedding\', \'detection\', \'segmentation\'):\n        cfg = config[group]\n        name = \'model.safetensors.index.json\' if group == \'embedding\' else \'model.safetensors\'\n        # Explicit anonymous access: no Hugging Face secret required.\n        get_hf_file_metadata(hf_hub_url(cfg[\'model_id\'], name, revision=cfg[\'revision\']), token=False)\n    return gpu_report()\n\n\ndef run_all(root, out, resume=None):\n    config = json.loads((root / \'config.json\').read_text(encoding=\'utf-8\'))\n    rows = load_rows(root)\n    expected = config[\'expected_images\']\n    if len(rows) != expected:\n        raise ValueError(f\'Expected {expected} images, got {len(rows)}\')\n    code_hashes = {name: digest_file(Path(__file__).with_name(name)) for name in (\'pipeline.py\', \'common.py\')}\n    identity = fingerprint({\'config\': config, \'manifest\': digest_file(root / \'manifest.jsonl\'), \'code\': code_hashes})\n    out.mkdir(parents=True, exist_ok=True)\n    if resume:\n        previous = json.loads((resume / \'run_manifest.json\').read_text(encoding=\'utf-8\'))\n        if previous.get(\'fingerprint\') != identity:\n            raise ValueError(\'Resume dataset/config fingerprint mismatch\')\n        for directory in (\'checkpoints\', \'masks\'):\n            if (resume / directory).exists():\n                shutil.copytree(resume / directory, out / directory, dirs_exist_ok=True)\n    existing = out / \'run_manifest.json\'\n    if existing.exists() and json.loads(existing.read_text(encoding=\'utf-8\')).get(\'fingerprint\') != identity:\n        raise ValueError(\'Output directory belongs to a different dataset/config\')\n    manifest = {\'schema_version\': 1, \'status\': \'preflight\', \'fingerprint\': identity,\n                \'config\': config, \'code_sha256\': code_hashes, \'images_total\': len(rows), \'images_completed\': 0,\n                \'images_failed\': 0, \'images_pending\': len(rows), \'errors\': [],\n                \'started_at_utc\': time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()),\n                \'packages\': {name: importlib.metadata.version(name) for name in\n                             [\'torch\', \'torchvision\', \'transformers\', \'accelerate\', \'numpy\', \'pyarrow\', \'qwen-vl-utils\']}}\n    atomic_json(existing, manifest)\n    try:\n        manifest[\'gpus\'] = preflight(config)\n        for i, row in enumerate(rows):\n            if digest_file(root / row[\'path\']) != row[\'sha256\']:\n                raise ValueError(f\'Image checksum mismatch: {row["imageID"]}\')\n            with Image.open(root / row[\'path\']) as image:\n                image.load()\n            if i % 1000 == 0:\n                print(f\'Validated input {i}/{len(rows)}\', flush=True)\n        write_parquet(out / \'images.parquet\', rows)\n        atomic_json(out / \'config.json\', config)\n        shutil.copy2(root / \'manifest.jsonl\', out / \'manifest.jsonl\')\n        for stage, smoke in [(\'embedding\', True), (\'objects\', True), (\'embedding\', False), (\'objects\', False)]:\n            manifest[\'status\'] = (\'smoke_\' if smoke else \'full_\') + stage\n            atomic_json(existing, manifest)\n            command = [sys.executable, \'-u\', str(Path(__file__).resolve()), \'--root\', str(root), \'--out\', str(out), \'--worker\', stage]\n            if smoke:\n                command.append(\'--smoke\')\n            (out / \'worker_error.json\').unlink(missing_ok=True)\n            # Separate processes guarantee GPU memory is released between models.\n            subprocess.run(command, check=True)\n        validate_vectors(np.load(out / \'image_embeddings.npy\', mmap_mode=\'r\'), len(rows), config[\'embedding\'][\'dimension\'])\n        import pyarrow.parquet as pq\n        if json.loads((out / \'embedding_ids.json\').read_text()) != [r[\'imageID\'] for r in rows]:\n            raise RuntimeError(\'Final embedding ID order mismatch\')\n        manifest.update(status=\'complete\', images_completed=len(rows), images_pending=0,\n                        object_instances=pq.read_metadata(out / \'objects.parquet\').num_rows,\n                        outputs={name: digest_file(out / name) for name in [\'images.parquet\', \'image_embeddings.npy\', \'embedding_ids.json\', \'objects.parquet\']})\n    except Exception as exc:\n        embedding_done = len(list((out / \'checkpoints\' / \'embeddings\').glob(\'*.npz\')))\n        objects_done = len(list((out / \'checkpoints\' / \'objects\').glob(\'*.json\')))\n        completed = len(set(p.stem for p in (out / \'checkpoints\' / \'embeddings\').glob(\'*.npz\')) &\n                        set(p.stem for p in (out / \'checkpoints\' / \'objects\').glob(\'*.json\')))\n        manifest.update(status=\'failed\', failed_stage=manifest[\'status\'], images_completed=completed,\n                        images_pending=len(rows)-completed, embedding_checkpoints=embedding_done,\n                        object_checkpoints=objects_done,\n                        errors=[{\'type\': type(exc).__name__, \'message\': str(exc)}])\n        if (out / \'worker_error.json\').exists():\n            detail = json.loads((out / \'worker_error.json\').read_text(encoding=\'utf-8\'))\n            manifest[\'errors\'].append(detail)\n            manifest[\'images_failed\'] = int(detail.get(\'imageID\') is not None)\n            manifest[\'images_pending\'] = len(rows) - completed - manifest[\'images_failed\']\n        atomic_json(existing, manifest)\n        raise\n    finally:\n        manifest[\'updated_at_utc\'] = time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime())\n        atomic_json(existing, manifest)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--root\', type=Path, required=True)\n    parser.add_argument(\'--out\', type=Path, required=True)\n    parser.add_argument(\'--resume\', type=Path)\n    parser.add_argument(\'--worker\', choices=[\'embedding\', \'objects\'])\n    parser.add_argument(\'--smoke\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.worker:\n        cfg = json.loads((args.root / \'config.json\').read_text(encoding=\'utf-8\'))\n        rows = load_rows(args.root)\n        worker = embedding_worker if args.worker == \'embedding\' else object_worker\n        try:\n            worker(args.root, args.out, rows, cfg, args.smoke)\n        except Exception as exc:\n            atomic_json(args.out / \'worker_error.json\', {\n                \'type\': type(exc).__name__, \'message\': str(exc), \'imageID\': ACTIVE_IMAGE,\n                \'stage\': args.worker, \'smoke\': args.smoke})\n            raise\n    else:\n        run_all(args.root, args.out, args.resume)\n', encoding='utf-8')
(CODE / 'requirements.txt').write_text('# This file was autogenerated by uv via the following command:\n#    uv pip compile source/image_retrieval/requirements-kaggle.txt --python-version 3.12 --python-platform x86_64-manylinux_2_28 --output-file source/image_retrieval/.work/requirements-kaggle.lock.txt --no-annotate\naccelerate==1.12.0\nav==19.0.2\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\neinops==0.8.1\nfilelock==4.2.4\nfsspec==2026.9.0\nhf-xet==1.7.0\nhuggingface-hub==0.36.2\nidna==3.20\njinja2==3.1.6\nmarkupsafe==3.0.4\nmpmath==1.3.0\nnetworkx==3.7\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.4.1\nnvidia-cuda-cupti-cu12==12.8.90\nnvidia-cuda-nvrtc-cu12==12.8.93\nnvidia-cuda-runtime-cu12==12.8.90\nnvidia-cudnn-cu12==9.10.2.21\nnvidia-cufft-cu12==11.3.3.83\nnvidia-cufile-cu12==1.13.1.3\nnvidia-curand-cu12==10.3.9.90\nnvidia-cusolver-cu12==11.7.3.90\nnvidia-cusparse-cu12==12.5.8.93\nnvidia-cusparselt-cu12==0.7.1\nnvidia-nccl-cu12==2.27.3\nnvidia-nvjitlink-cu12==12.8.93\nnvidia-nvtx-cu12==12.8.90\npackaging==26.3\npillow==12.1.0\npsutil==7.2.2\npyarrow==23.0.1\npyyaml==6.0.3\nqwen-vl-utils==0.0.14\nregex==2026.9.29\nrequests==2.34.2\nsafetensors==0.7.0\nsetuptools==84.2.1\nsympy==1.14.0\ntokenizers==0.22.2\ntorch==2.8.0\ntorchvision==0.23.0\ntqdm==4.70.1\ntransformers==4.57.6\ntriton==3.4.0\ntyping-extensions==4.16.0\nurllib3==2.8.0\n', encoding='utf-8')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.9.7'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/tmp/image-retrieval-env'], check=True)
    PYTHON = '/tmp/image-retrieval-env/bin/python'
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                    '-r', str(CODE / 'requirements.txt')], check=True)
except Exception as exc:
    (OUT / 'run_manifest.json').write_text(json.dumps({'status': 'failed', 'failed_stage': 'bootstrap',
        'images_total': 5000, 'images_completed': 0, 'images_failed': 0, 'images_pending': 5000,
        'errors': [{'type': type(exc).__name__, 'message': str(exc)}]}, indent=2))
    raise


In [ ]:
# Optional: mount output from an earlier run and set its output directory here.
RESUME_DIR = ''
import zipfile
roots = list(pathlib.Path('/kaggle/input').rglob('manifest.jsonl'))
roots = [p.parent for p in roots if (p.parent / 'config.json').exists() and
         ((p.parent / 'images').exists() or (p.parent / 'images.zip').exists())]
if len(roots) != 1:
    raise RuntimeError(f'Expected exactly one input dataset with images/manifest; found {roots}')
DATA = roots[0]
if not (DATA / 'images').exists() and (DATA / 'images.zip').exists():
    target = pathlib.Path('/tmp/image_retrieval_input')
    target.mkdir(exist_ok=True)
    for name in ('manifest.jsonl', 'config.json'):
        import shutil
        shutil.copy2(DATA / name, target / name)
    with zipfile.ZipFile(DATA / 'images.zip') as archive:
        for member in archive.infolist():
            path = (target / member.filename).resolve()
            if not path.is_relative_to(target.resolve()):
                raise RuntimeError('Unsafe archive path')
        names = archive.namelist()
        destination = target if any(name.startswith('images/') for name in names) else target / 'images'
        archive.extractall(destination)
    DATA = target
command = [PYTHON, '-u', str(CODE / 'pipeline.py'), '--root', str(DATA), '--out', str(OUT)]
if RESUME_DIR:
    command += ['--resume', RESUME_DIR]
# These checkpoints are public. Deliberately do not read Kaggle/Hugging Face secrets.
env = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', TOKENIZERS_PARALLELISM='false')
env.pop('HF_TOKEN', None)
env.pop('HUGGING_FACE_HUB_TOKEN', None)
with (OUT / 'pipeline.log').open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    code = process.wait()
if code:
    raise RuntimeError(f'Pipeline failed (exit {code}); inspect run_manifest.json and pipeline.log')
display(json.loads((OUT / 'run_manifest.json').read_text()))


In [ ]:
from IPython.display import display, Image
for path in sorted((OUT / 'smoke_overlays').glob('*.jpg')):
    print(path.name)
    display(Image(filename=str(path)))
